In [19]:
from __future__ import annotations

import os
import traceback
from pathlib import Path
from typing import Iterable

import pandas as pd
import soundfile as sf
from pyannote.audio import Pipeline
import torch
from pyannote.core import Annotation, Segment

In [20]:
import pyannote.audio
print(pyannote.audio.__version__)

4.0.7


In [22]:
# ============================================================
# CONFIGURATION
# ============================================================

# Root directory containing participant folders.
#
# Example:
# Processed Dataset/
#     300/
#         300_AUDIO_removed_interviewer.wav
#     301/
#         301_AUDIO_removed_interviewer.wav
PROCESSED_DATASET_DIRECTORY = Path("/Volumes/LaCie/DAIC-WOZ/Processed Dataset")

# Input WAV files to process.
INPUT_AUDIO_SUFFIX = "_removed_interviewer.wav"

# CSV containing one row for every candidate overlap event.
OUTPUT_EVENTS_CSV = (
    PROCESSED_DATASET_DIRECTORY
    / "acoustic_overlap_events.csv"
)

# CSV containing one summary row for every recording.
OUTPUT_SUMMARY_CSV = (
    PROCESSED_DATASET_DIRECTORY
    / "acoustic_overlap_summary.csv"
)

# Modern local pyannote diarization pipeline.
MODEL_NAME = "pyannote/speaker-diarization-community-1"

# Ignore candidate overlap regions shorter than this duration.
MINIMUM_OVERLAP_DURATION_SECONDS = 0.20

# Combine nearby overlap regions when the gap between them is no
# larger than this value.
MERGE_GAP_SECONDS = 0.15

# Include this much audio before and after the detected interval
# for manual listening.
LISTENING_CONTEXT_SECONDS = 1.00

# Save progress after this many recordings.
SAVE_EVERY_N_RECORDINGS = 10

# Set to True to skip recordings already listed as successfully
# processed in the summary CSV.
RESUME_PROCESSING = True

DEVICE = "auto"


# ============================================================
# CSV COLUMNS
# ============================================================

EVENT_COLUMNS = [
    "participant_id",
    "audio_filename",
    "audio_path",
    "event_number",
    "start_seconds",
    "stop_seconds",
    "duration_seconds",
    "start_time",
    "stop_time",
    "review_start_seconds",
    "review_stop_seconds",
    "review_start_time",
    "review_stop_time",
    "audio_duration_seconds",
    "manual_label",
    "notes",
]

SUMMARY_COLUMNS = [
    "participant_id",
    "audio_filename",
    "audio_path",
    "audio_duration_seconds",
    "number_of_overlap_events",
    "total_overlap_duration_seconds",
    "percent_overlap",
    "status",
    "error",
]


# ============================================================
# TIME FORMATTING
# ============================================================

def seconds_to_timestamp(seconds: float) -> str:
    """
    Convert seconds to HH:MM:SS.mmm.

    Examples
    --------
    134.52 -> "00:02:14.520"
    """
    seconds = max(0.0, float(seconds))

    hours = int(seconds // 3600)
    remaining_seconds = seconds % 3600

    minutes = int(remaining_seconds // 60)
    remaining_seconds %= 60

    whole_seconds = int(remaining_seconds)
    milliseconds = int(
        round((remaining_seconds - whole_seconds) * 1000)
    )

    # Handle rounding such as 59.9996 -> 60.000.
    if milliseconds == 1000:
        milliseconds = 0
        whole_seconds += 1

    if whole_seconds == 60:
        whole_seconds = 0
        minutes += 1

    if minutes == 60:
        minutes = 0
        hours += 1

    return (
        f"{hours:02d}:"
        f"{minutes:02d}:"
        f"{whole_seconds:02d}."
        f"{milliseconds:03d}"
    )


# ============================================================
# FILE DISCOVERY
# ============================================================

def find_audio_files(
    root_directory: Path,
    audio_suffix: str,
) -> list[Path]:
    """
    Recursively find WAV files ending with the requested suffix.
    """
    if not root_directory.exists():
        raise FileNotFoundError(
            f"Dataset directory does not exist: "
            f"{root_directory}"
        )

    if not root_directory.is_dir():
        raise NotADirectoryError(
            f"Dataset path is not a directory: "
            f"{root_directory}"
        )

    return sorted(
        path
        for path in root_directory.rglob(f"*{audio_suffix}")
        if path.is_file()
    )


def get_participant_id(audio_path: Path) -> str:
    """
    Infer the participant ID from the parent folder.

    Example
    -------
    Processed Dataset/300/300_AUDIO_removed_interviewer.wav
    returns "300".
    """
    return audio_path.parent.name


# ============================================================
# DEVICE SELECTION
# ============================================================

def select_device(device_name: str = "auto") -> torch.device:
    """
    Select a PyTorch device.
    """
    normalized_name = device_name.strip().lower()

    if normalized_name == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")

        if (
            hasattr(torch.backends, "mps")
            and torch.backends.mps.is_available()
        ):
            return torch.device("mps")

        return torch.device("cpu")

    if normalized_name == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError(
                "DEVICE='cuda' was requested, but CUDA is not "
                "available."
            )

        return torch.device("cuda")

    if normalized_name == "mps":
        if not (
            hasattr(torch.backends, "mps")
            and torch.backends.mps.is_available()
        ):
            raise RuntimeError(
                "DEVICE='mps' was requested, but Apple MPS is not "
                "available."
            )

        return torch.device("mps")

    if normalized_name == "cpu":
        return torch.device("cpu")

    raise ValueError(
        f"Unsupported DEVICE value: {device_name!r}. "
        "Use 'auto', 'cpu', 'mps', or 'cuda'."
    )


# ============================================================
# PIPELINE LOADING
# ============================================================

def load_diarization_pipeline() -> Pipeline:
    """
    Load the modern pyannote speaker diarization pipeline.

    The Hugging Face access token must be available through the
    HF_TOKEN environment variable.
    """
    token = os.getenv("HF_TOKEN")

    if not token:
        raise RuntimeError(
            "The HF_TOKEN environment variable is not set. "
            "Set HF_TOKEN to a valid Hugging Face access token "
            "before running this script."
        )

    try:
        pipeline = Pipeline.from_pretrained(
            MODEL_NAME,
            token=token,
        )

    except TypeError as error:
        raise RuntimeError(
            f"Could not load '{MODEL_NAME}' because the installed "
            "pyannote.audio API is incompatible with this script. "
            "This script expects pyannote.audio 4.x and uses the "
            "'token' argument."
        ) from error

    except Exception as error:
        raise RuntimeError(
            f"Could not load the pyannote pipeline '{MODEL_NAME}'. "
            "Confirm that your Hugging Face token is valid and that "
            "you accepted the model's access conditions."
        ) from error

    if pipeline is None:
        raise RuntimeError(
            f"Pipeline.from_pretrained returned None for "
            f"'{MODEL_NAME}'."
        )

    device = select_device(DEVICE)

    try:
        pipeline.to(device)
    except Exception as error:
        if device.type == "mps":
            print(
                "Warning: the pipeline could not be moved to MPS. "
                "Falling back to CPU."
            )
            print(f"MPS error: {error}")
            device = torch.device("cpu")
            pipeline.to(device)
        else:
            raise RuntimeError(
                f"Could not move the pipeline to {device}."
            ) from error

    print(f"Loaded model: {MODEL_NAME}")
    print(f"Using device: {device}")

    return pipeline


# ============================================================
# PYANNOTE OUTPUT HANDLING
# ============================================================

def extract_speaker_annotation(
    pipeline_output: object,
) -> Annotation:
    """
    Extract the regular speaker diarization Annotation from a
    pyannote 4 pipeline result.

    pyannote.audio 4 pipelines generally return an output object
    containing a `speaker_diarization` attribute. This function
    also allows an Annotation directly for compatibility.
    """
    if isinstance(pipeline_output, Annotation):
        return pipeline_output

    speaker_diarization = getattr(
        pipeline_output,
        "speaker_diarization",
        None,
    )

    if isinstance(speaker_diarization, Annotation):
        return speaker_diarization

    raise TypeError(
        "The pyannote pipeline output did not contain a valid "
        "'speaker_diarization' Annotation. "
        f"Received output type: {type(pipeline_output).__name__}"
    )


def annotation_to_overlap_intervals(
    annotation: Annotation,
) -> list[tuple[float, float]]:
    """
    Find regions where at least two predicted speakers are active.

    Returns
    -------
    list[tuple[float, float]]
        A list of (start_seconds, stop_seconds) intervals.
    """
    overlap_timeline = annotation.get_overlap()

    intervals: list[tuple[float, float]] = []

    for segment in overlap_timeline:
        start = float(segment.start)
        stop = float(segment.end)

        if stop > start:
            intervals.append((start, stop))

    return intervals


# ============================================================
# INTERVAL PROCESSING
# ============================================================

def merge_intervals(
    intervals: Iterable[tuple[float, float]],
    maximum_gap_seconds: float,
) -> list[tuple[float, float]]:
    """
    Merge intervals that overlap or are separated by a small gap.
    """
    sorted_intervals = sorted(
        (float(start), float(stop))
        for start, stop in intervals
        if float(stop) > float(start)
    )

    if not sorted_intervals:
        return []

    merged: list[list[float]] = [
        [
            sorted_intervals[0][0],
            sorted_intervals[0][1],
        ]
    ]

    for current_start, current_stop in sorted_intervals[1:]:
        previous_start, previous_stop = merged[-1]

        gap = current_start - previous_stop

        if gap <= maximum_gap_seconds:
            merged[-1][1] = max(
                previous_stop,
                current_stop,
            )
        else:
            merged.append(
                [current_start, current_stop]
            )

    return [
        (start, stop)
        for start, stop in merged
    ]


def clip_intervals(
    intervals: Iterable[tuple[float, float]],
    audio_duration_seconds: float,
) -> list[tuple[float, float]]:
    """
    Restrict intervals to valid audio boundaries.
    """
    clipped: list[tuple[float, float]] = []

    for start, stop in intervals:
        clipped_start = max(
            0.0,
            min(float(start), audio_duration_seconds),
        )

        clipped_stop = max(
            0.0,
            min(float(stop), audio_duration_seconds),
        )

        if clipped_stop > clipped_start:
            clipped.append(
                (clipped_start, clipped_stop)
            )

    return clipped


def filter_short_intervals(
    intervals: Iterable[tuple[float, float]],
    minimum_duration_seconds: float,
) -> list[tuple[float, float]]:
    """
    Remove intervals shorter than the configured minimum.
    """
    return [
        (start, stop)
        for start, stop in intervals
        if stop - start >= minimum_duration_seconds
    ]


# ============================================================
# AUDIO METADATA
# ============================================================

def get_audio_duration(audio_path: Path) -> float:
    """
    Read WAV duration using soundfile.
    """
    try:
        audio_information = sf.info(str(audio_path))

    except Exception as error:
        raise RuntimeError(
            f"Could not read audio metadata for: {audio_path}"
        ) from error

    if audio_information.samplerate <= 0:
        raise ValueError(
            f"Invalid sample rate for: {audio_path}"
        )

    duration = (
        audio_information.frames
        / audio_information.samplerate
    )

    return float(duration)


# ============================================================
# RECORDING PROCESSING
# ============================================================

def process_recording(
    pipeline: Pipeline,
    audio_path: Path,
) -> tuple[list[dict[str, object]], dict[str, object]]:
    """
    Process one recording and generate event and summary rows.
    """
    participant_id = get_participant_id(audio_path)
    audio_duration = get_audio_duration(audio_path)

    print(
        f"Processing participant {participant_id}: "
        f"{audio_path.name}"
    )

    pipeline_output = pipeline(str(audio_path))

    speaker_annotation = extract_speaker_annotation(
        pipeline_output
    )

    overlap_intervals = annotation_to_overlap_intervals(
        speaker_annotation
    )

    overlap_intervals = clip_intervals(
        overlap_intervals,
        audio_duration_seconds=audio_duration,
    )

    overlap_intervals = merge_intervals(
        overlap_intervals,
        maximum_gap_seconds=MERGE_GAP_SECONDS,
    )

    overlap_intervals = filter_short_intervals(
        overlap_intervals,
        minimum_duration_seconds=(
            MINIMUM_OVERLAP_DURATION_SECONDS
        ),
    )

    event_rows: list[dict[str, object]] = []

    for event_number, (start, stop) in enumerate(
        overlap_intervals,
        start=1,
    ):
        duration = stop - start

        review_start = max(
            0.0,
            start - LISTENING_CONTEXT_SECONDS,
        )

        review_stop = min(
            audio_duration,
            stop + LISTENING_CONTEXT_SECONDS,
        )

        event_rows.append(
            {
                "participant_id": participant_id,
                "audio_filename": audio_path.name,
                "audio_path": str(audio_path.resolve()),
                "event_number": event_number,
                "start_seconds": round(start, 3),
                "stop_seconds": round(stop, 3),
                "duration_seconds": round(duration, 3),
                "start_time": seconds_to_timestamp(start),
                "stop_time": seconds_to_timestamp(stop),
                "review_start_seconds": round(
                    review_start,
                    3,
                ),
                "review_stop_seconds": round(
                    review_stop,
                    3,
                ),
                "review_start_time": seconds_to_timestamp(
                    review_start
                ),
                "review_stop_time": seconds_to_timestamp(
                    review_stop
                ),
                "audio_duration_seconds": round(
                    audio_duration,
                    3,
                ),
                "manual_label": "",
                "notes": "",
            }
        )

    total_overlap_duration = sum(
        stop - start
        for start, stop in overlap_intervals
    )

    if audio_duration > 0:
        percent_overlap = (
            total_overlap_duration
            / audio_duration
            * 100
        )
    else:
        percent_overlap = 0.0

    summary_row: dict[str, object] = {
        "participant_id": participant_id,
        "audio_filename": audio_path.name,
        "audio_path": str(audio_path.resolve()),
        "audio_duration_seconds": round(
            audio_duration,
            3,
        ),
        "number_of_overlap_events": len(
            overlap_intervals
        ),
        "total_overlap_duration_seconds": round(
            total_overlap_duration,
            3,
        ),
        "percent_overlap": round(
            percent_overlap,
            3,
        ),
        "status": "success",
        "error": "",
    }

    return event_rows, summary_row


# ============================================================
# CSV UTILITIES
# ============================================================

def load_existing_csv(
    csv_path: Path,
    columns: list[str],
) -> pd.DataFrame:
    """
    Load an existing progress CSV or create an empty DataFrame.
    """
    if not csv_path.exists():
        return pd.DataFrame(columns=columns)

    try:
        dataframe = pd.read_csv(
            csv_path,
            dtype={
                "participant_id": "string",
                "audio_filename": "string",
                "audio_path": "string",
            },
        )

    except pd.errors.EmptyDataError:
        return pd.DataFrame(columns=columns)

    except Exception as error:
        raise RuntimeError(
            f"Could not read existing CSV: {csv_path}"
        ) from error

    for column in columns:
        if column not in dataframe.columns:
            dataframe[column] = pd.NA

    return dataframe[columns]


def save_progress(
    event_rows: list[dict[str, object]],
    summary_rows: list[dict[str, object]],
) -> None:
    """
    Save all current event and summary records.
    """
    OUTPUT_EVENTS_CSV.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    event_dataframe = pd.DataFrame(
        event_rows,
        columns=EVENT_COLUMNS,
    )

    summary_dataframe = pd.DataFrame(
        summary_rows,
        columns=SUMMARY_COLUMNS,
    )

    event_dataframe.to_csv(
        OUTPUT_EVENTS_CSV,
        index=False,
    )

    summary_dataframe.to_csv(
        OUTPUT_SUMMARY_CSV,
        index=False,
    )


def get_successfully_processed_paths(
    summary_dataframe: pd.DataFrame,
) -> set[str]:
    """
    Return resolved audio paths already processed successfully.
    """
    if summary_dataframe.empty:
        return set()

    required_columns = {
        "audio_path",
        "status",
    }

    if not required_columns.issubset(
        summary_dataframe.columns
    ):
        return set()

    successful_rows = summary_dataframe[
        summary_dataframe["status"].eq("success")
    ]

    return {
        str(Path(path).resolve())
        for path in successful_rows["audio_path"].dropna()
    }


# ============================================================
# DATASET PROCESSING
# ============================================================

def process_dataset() -> None:
    """
    Process all matching WAV files and write overlap-review CSVs.
    """
    audio_paths = find_audio_files(
        PROCESSED_DATASET_DIRECTORY,
        INPUT_AUDIO_SUFFIX,
    )

    if not audio_paths:
        print(
            "No matching audio files were found under:"
        )
        print(PROCESSED_DATASET_DIRECTORY)
        return

    print(
        f"Found {len(audio_paths)} matching recordings."
    )

    existing_events_dataframe = load_existing_csv(
        OUTPUT_EVENTS_CSV,
        EVENT_COLUMNS,
    )

    existing_summary_dataframe = load_existing_csv(
        OUTPUT_SUMMARY_CSV,
        SUMMARY_COLUMNS,
    )

    all_event_rows = existing_events_dataframe.to_dict(
        orient="records"
    )

    all_summary_rows = existing_summary_dataframe.to_dict(
        orient="records"
    )

    successfully_processed_paths: set[str] = set()

    if RESUME_PROCESSING:
        successfully_processed_paths = (
            get_successfully_processed_paths(
                existing_summary_dataframe
            )
        )

    remaining_audio_paths = [
        audio_path
        for audio_path in audio_paths
        if str(audio_path.resolve())
        not in successfully_processed_paths
    ]

    print(
        f"Already completed: "
        f"{len(audio_paths) - len(remaining_audio_paths)}"
    )
    print(
        f"Remaining recordings: "
        f"{len(remaining_audio_paths)}"
    )

    if not remaining_audio_paths:
        print("No recordings remain.")
        return

    pipeline = load_diarization_pipeline()

    successful_recordings = 0
    failed_recordings = 0

    try:
        for recording_index, audio_path in enumerate(
            remaining_audio_paths,
            start=1,
        ):
            participant_id = get_participant_id(
                audio_path
            )

            try:
                event_rows, summary_row = (
                    process_recording(
                        pipeline=pipeline,
                        audio_path=audio_path,
                    )
                )

                all_event_rows.extend(event_rows)
                all_summary_rows.append(summary_row)

                successful_recordings += 1

                print(
                    f"  Candidate overlap events: "
                    f"{summary_row['number_of_overlap_events']}"
                )

            except Exception as error:
                failed_recordings += 1

                error_message = (
                    f"{type(error).__name__}: {error}"
                )

                print(
                    f"  Failed: {error_message}"
                )

                traceback.print_exc()

                try:
                    duration = get_audio_duration(
                        audio_path
                    )
                except Exception:
                    duration = pd.NA

                all_summary_rows.append(
                    {
                        "participant_id": participant_id,
                        "audio_filename": audio_path.name,
                        "audio_path": str(
                            audio_path.resolve()
                        ),
                        "audio_duration_seconds": duration,
                        "number_of_overlap_events": pd.NA,
                        "total_overlap_duration_seconds": pd.NA,
                        "percent_overlap": pd.NA,
                        "status": "failed",
                        "error": error_message,
                    }
                )

            if (
                recording_index
                % SAVE_EVERY_N_RECORDINGS
                == 0
            ):
                save_progress(
                    all_event_rows,
                    all_summary_rows,
                )

                print(
                    f"Saved progress after "
                    f"{recording_index} recordings."
                )

    except KeyboardInterrupt:
        print(
            "\nProcessing interrupted. Saving current "
            "progress before exiting."
        )

        save_progress(
            all_event_rows,
            all_summary_rows,
        )

        raise

    finally:
        save_progress(
            all_event_rows,
            all_summary_rows,
        )

    print("\nProcessing complete.")
    print(
        f"Successful recordings: "
        f"{successful_recordings}"
    )
    print(
        f"Failed recordings: "
        f"{failed_recordings}"
    )
    print(
        f"Overlap events CSV: "
        f"{OUTPUT_EVENTS_CSV}"
    )
    print(
        f"Recording summary CSV: "
        f"{OUTPUT_SUMMARY_CSV}"
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    process_dataset()

Found 186 matching recordings.
Already completed: 0
Remaining recordings: 186


config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 5.91MB            

segmentation/pytorch_model.bin: downloading bytes:           |  0.00B            

plda/xvec_transform.npz: reconstructing file:   0%|          |  0.00B /  134kB            

...

plda/plda.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/plda.npz: downloading bytes:           |  0.00B            

embedding/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 26.6MB            

embedding/pytorch_model.bin: downloading bytes:           |  0.00B            

Loaded model: pyannote/speaker-diarization-community-1
Using device: mps
Processing participant 300_P: 300_AUDIO_removed_interviewer.wav
  Candidate overlap events: 0
Processing participant 301_P: 301_AUDIO_removed_interviewer.wav
  Candidate overlap events: 3
Processing participant 302_P: 302_AUDIO_removed_interviewer.wav
  Candidate overlap events: 0
Processing participant 303_P: 303_AUDIO_removed_interviewer.wav
  Candidate overlap events: 1
Processing participant 304_P: 304_AUDIO_removed_interviewer.wav
  Candidate overlap events: 4
Processing participant 305_P: 305_AUDIO_removed_interviewer.wav
  Candidate overlap events: 11
Processing participant 306_P: 306_AUDIO_removed_interviewer.wav
  Candidate overlap events: 5
Processing participant 307_P: 307_AUDIO_removed_interviewer.wav
  Candidate overlap events: 3
Processing participant 308_P: 308_AUDIO_removed_interviewer.wav
  Candidate overlap events: 1
Processing participant 309_P: 309_AUDIO_removed_interviewer.wav
  Candidate over